In [8]:
# ============================================================
# RELATÓRIO DE PASSIVO — envio por E-MAIL + WHATSAPP (GARE11 e GAME11)
# ============================================================
import os
import smtplib
from email.mime.multipart import MIMEMultipart
from email.mime.text import MIMEText
from email.mime.base import MIMEBase
from email import encoders
from datetime import datetime
from dotenv import load_dotenv

# Helpers Z-API já usados pelo relatório ATIVO — importados, não duplicados.
from resumo_ativo import enviar_texto_whatsapp, enviar_documento_whatsapp

load_dotenv()

# Grupo "Guardian - PASSIVO inteligência listados" (formato: <id>-group).
GRUPO_PASSIVO_PHONE = os.getenv("GRUPO_PASSIVO_PHONE")

In [9]:
# ============================================================
# 1) DATA E CAMINHOS DOS PDFs — único lugar onde a data é editada
# ============================================================
data = "04.09.2026"   # ou: datetime.today().strftime("%d.%m.%Y")

DIARIOS = (
    r"g:\.shortcut-targets-by-id\1YZLLTFOuLniE6hO7WNHXl2c3MSa4TOv6\X. GAME11"
    r"\GUARDIAN Institucional\5. Planejamento & Processos\2. 2023\4. Research\2025"
    r"\5. Inteligência Passivos Guardian\2. RELATÓRIOS\1. DIARIOS"
)

# Atenção: o GARE exporta "Passivos_" (plural) e o GAME "Passivo_" (singular).
pdf_gare = rf"{DIARIOS}\GARE\Passivos_GARE11_{data}.pdf"
pdf_game = rf"{DIARIOS}\GAME\Passivo_GAME11_{data}.pdf"

In [10]:
# ============================================================
# 2) TEXTOS DO DIA ✍️ — a ÚNICA célula que muda todo dia
# ------------------------------------------------------------
#   CORPO_*  -> miolo HTML do e-mail (só as tags <p>; o invólucro
#               <html><body> e a assinatura entram em _html_email, célula 3).
#               Em HTML, use &gt; e &lt; no lugar de > e < (ex.: &gt;15M).
#   WHATS_*  -> texto do WhatsApp, enviado VERBATIM (emojis e "> " normais).
# ============================================================

# ---------------------------- GARE11 ----------------------------
CORPO_GARE = """
<p>Olá, time,</p>

<p>Resumo do cenário diário GARE11 — <b>04/09/2026</b>: cota em queda de <b>-0,48%</b>, fechando a <b>R$ 8,34</b>, com Tivio Opportunities mantendo pressão vendedora na faixa 3M-15M — maior venda do dia. Relatório completo em anexo.</p>

<p><b>📊 Desempenho do Dia</b><br>
GARE11: -0,48% | IFIX: 0,2% | Daytrade: 42% do volume total</p>

<p><b>📈 Compras</b> (TOP 10 – R$ 0,9 mi)<br>
5k-20k: 30,79% | 1k-5k: 18,96% | 0-1k: 16,26%</p>

<p>Kinea Hedge Fund comprou R$ 416k mantendo-se na faixa 500k-1M — maior compra do dia. Ao todo, 5 novos cotistas (NC) ingressaram no fundo totalizando R$ 255k, com destaque para Alan Nader Ackel Ghani com R$ 100k. Cotista da faixa 100k-500k comprou R$ 97k mantendo-se na mesma faixa.</p>

<p><b>📉 Vendas</b> (TOP 10 – R$ 2,6 mi)<br>
3M-15M: 38,97% | 5k-20k: 14,27% | 50k-100k: 12,67%</p>

<p>Tivio Opportunities Master Fundo vendeu R$ 1,6 mi mantendo-se na faixa 3M-15M — maior venda do dia. XTX Markets Trading Limited reduziu posição de R$ 328k saindo de 100k-500k para 20k-50k. Cotistas da faixa 50k-100k zeraram posições totalizando R$ 337k.</p>
"""

WHATS_GARE = """📅 GARE11 - 04/09/2026

⚠️ NECTON LIDERA SALDO NEGATIVO EM DIA DE QUEDA DA COTA CONTRA O IFIX COM TIVIO OPPORTUNITIES MANTENDO PRESSÃO VENDEDORA NA FAIXA 3M-15M

📈 Volume negociado foi de R$ 7,1 milhões. A cota fechou o dia em queda de -0,48%, fechando a R$ 8,34. O fundo registrou mais um dia de entrada líquida de cotistas, ultrapassando o patamar de 555 mil

📊 Desempenho do Dia:
* GARE11: -0,48%
* IFIX: 0,2%
* Daytrade: 42% do volume total

📈 Compras (TOP 10 – R$ 0,9 mi)
* 5k-20k: 30,79%
* 1k-5k: 18,96%
* 0-1k: 16,26%

🔹KINEA HEDGE FUND compra R$ 416k mantendo-se na faixa 500k-1M — maior compra do dia.
🔹5 novos cotistas (NC) ingressam no fundo totalizando R$ 255k, com destaque para ALAN NADER ACKEL GHANI com R$ 100k.
🔹Cotista da faixa 100k-500k compra R$ 97k mantendo-se na mesma faixa.

📉 Vendas (TOP 10 – R$ 2,6 mi)
* 3M-15M: 38,97%
* 5k-20k: 14,27%
* 50k-100k: 12,67%

🔹TIVIO OPPORTUNITIES MASTER FUNDO vende R$ 1,6 mi mantendo-se na faixa 3M-15M — maior venda do dia.
🔹XTX MARKETS TRADING LIMITED reduz posição de R$ 328k saindo de 100k-500k para 20k-50k.
🔹Cotistas da faixa 50k-100k zeram posições totalizando R$ 337k."""


# ---------------------------- GAME11 ----------------------------
# Mesma estrutura do GARE11 — preencher com os números do dia.
CORPO_GAME = """
<p>Olá, time,</p>

    <p>Resumo diário do fundo <b>GAME11</b> — <b>04/09/2026</b>.</p>

    <p><b>Destaque do dia:</b> GUARDIAN STRATEGY FII compra R$87k no sétimo pregão consecutivo, enquanto MANAT RECEBÍVEIS e XTX MARKETS somam R$137k em vendas institucionais.</p>

    <p>O volume negociado foi de <b>R$ 0,26 milhões</b>. A cota fechou o dia em <b>alta de 0,9%</b>, cotada a <b>R$ 8,29</b>. O fundo registrou <b>saída líquida</b> de cotistas (9 entradas x 16 saídas), mantendo-se no patamar de <b>20.244</b> cotistas.</p>

    <p><b>Desempenho do dia:</b><br>
    GAME11: 0,9%<br>
    IFIX PAPEL: 0,1%<br>
    Daytrade: 43% do volume total</p>

    <p><b>Compras (Top 10 – R$ 129 mil):</b><br>
    As compras se concentraram nas faixas <b>&gt;15M (59,27%)</b>, <b>5k-20k (17,51%)</b> e <b>20k-50k (11,00%)</b>. O GUARDIAN STRATEGY FII DE RL comprou R$87k, mantendo-se na faixa &gt;15M pelo sétimo pregão consecutivo; somado à RIGEL COVE LP, que adquiriu R$5k saindo da faixa 0-1k para 5k-20k, os institucionais responderam por R$92k das compras. Um cotista da faixa 20k-50k comprou R$11k e migrou para a faixa 50k-100k, entrando em categoria de alta concentração financeira. Um cotista da faixa 5k-20k comprou R$15k, maior movimentação de pessoa física do dia.</p>

    <p><b>Vendas (Top 10 – R$ 145 mil):</b><br>
    As vendas se concentraram nas faixas <b>1M-3M (84,66%)</b>, <b>100k-500k (9,01%)</b> e <b>0-1k (3,31%)</b>. O MANAT RECEBÍVEIS ESTRUTURADOS FII realizou venda expressiva de R$124k, mantendo-se na faixa 1M-3M, e a XTX MARKETS TRADING LIMITED vendeu R$13k na faixa 100k-500k, somando R$137k em vendas institucionais. Apenas um cotista, da faixa 1k-5k, zerou posição no dia, com R$1k. As vendas de pessoa física ficaram praticamente ausentes, somando R$6k pulverizados nas faixas 0-1k e 5k-20k.</p>
"""

WHATS_GAME = """📅 GAME11 - 04/09/2026

⚠️ GUARDIAN STRATEGY FII COMPRA R$87K NO SÉTIMO PREGÃO CONSECUTIVO, COM MANAT RECEBÍVEIS E XTX MARKETS SOMANDO R$137K EM VENDAS INSTITUCIONAIS

📈 Volume negociado foi de R$ 0,26 milhões. A cota fechou o dia em alta de 0,9%, fechando a R$ 8,29. O fundo registrou saída líquida de cotistas (9 entradas x 16 saídas), mantendo-se no patamar de 20.244 cotistas.

📊 Desempenho do Dia:

* GAME11: 0,9%
* IFIX PAPEL: 0,1%
* Daytrade: 43% do volume total

📈 Compras (TOP 10 – R$ 129 mil)

* >15M: 59,27%
* 5k-20k: 17,51%
* 20k-50k: 11,00%

🔹GUARDIAN STRATEGY FII DE RL compra R$87k, mantendo-se na faixa >15M pelo sétimo pregão consecutivo; somado à RIGEL COVE LP, que adquiriu R$5k saindo da faixa 0-1k para 5k-20k, os institucionais respondem por R$92k das compras.
🔹Cotista da faixa 20k-50k compra R$11k e migra para a faixa 50k-100k, entrando em categoria de alta concentração financeira.
🔹Cotista da faixa 5k-20k compra R$15k, maior movimentação de PF do dia.

📉 Vendas (TOP 10 – R$ 145 mil)

* 1M-3M: 84,66%
* 100k-500k: 9,01%
* 0-1k: 3,31%

🔹MANAT RECEBÍVEIS ESTRUTURADOS FII realiza venda expressiva de R$124k, mantendo-se na faixa 1M-3M, e a XTX MARKETS TRADING LIMITED vende R$13k na faixa 100k-500k, somando R$137k em vendas institucionais.
🔹Apenas um cotista, da faixa 1k-5k, zera posição no dia, com R$1k.
🔹As vendas de PF ficam praticamente ausentes, somando R$6k pulverizados nas faixas 0-1k e 5k-20k."""

In [11]:
# ============================================================
# 3) FUNÇÕES DE ENVIO — uma implementação, usada pelos dois fundos
# ============================================================
REMETENTE = "lc@guardian-asset.com"

DESTINATARIOS = [
    "lc@guardian-asset.com", "pmk@guardian-asset.com", "msf@guardian-asset.com",
    "cm@guardian-asset.com", "lb@guardian-asset.com", "ja@guardian-asset.com",
    "gsa@guardian-asset.com", "gvm@guardian-asset.com", "rt@guardian-asset.com",
    "acl@guardian-asset.com", "lt@guardian-asset.com", "tfa@guardian-asset.com",
]

# Fecha todo e-mail, nos dois fundos (anexada por _html_email).
ASSINATURA = """<p>Att,<br>

Lucas Cavalcante.</p>"""


def _html_email(corpo):
    """Embrulha o miolo HTML no mesmo estilo usado no relatório ATIVO."""
    return (
        '<html><body style="font-family: Arial, sans-serif; font-size: 14px; '
        'color: #222222; line-height: 1.55;">'
        f"{corpo.strip()}"
        f"{ASSINATURA}"
        "</body></html>"
    )


def _enviar_email(pdf_path, corpo, assunto):
    msg = MIMEMultipart()
    msg["From"] = REMETENTE
    msg["To"] = ", ".join(DESTINATARIOS)
    msg["Subject"] = assunto
    msg.attach(MIMEText(_html_email(corpo), "html"))

    # Anexar o PDF
    try:
        with open(pdf_path, "rb") as attachment:
            part = MIMEBase("application", "octet-stream")
            part.set_payload(attachment.read())
        encoders.encode_base64(part)
        part.add_header(
            "Content-Disposition",
            f"attachment; filename= {os.path.basename(pdf_path)}",
        )
        msg.attach(part)
    except Exception as e:
        print(f"❌ Falha ao anexar o arquivo: {e}")
        return

    # Enviar via SMTP do Gmail
    try:
        with smtplib.SMTP_SSL("smtp.gmail.com", 465) as smtp_server:
            smtp_server.login(REMETENTE, os.getenv("senha_gmail"))
            smtp_server.sendmail(REMETENTE, DESTINATARIOS, msg.as_string())
        print(f"✅ E-mail enviado — {assunto}")
    except Exception as e:
        print(f"❌ Falha ao enviar e-mail: {e}")


def _enviar_whatsapp(texto, pdf_path):
    """Envia o texto e, em seguida, o PDF ao grupo do PASSIVO."""
    if not GRUPO_PASSIVO_PHONE:
        raise ValueError("GRUPO_PASSIVO_PHONE não definido no .env.")
    enviar_texto_whatsapp(texto, phone=GRUPO_PASSIVO_PHONE)
    enviar_documento_whatsapp(pdf_path, phone=GRUPO_PASSIVO_PHONE)


def enviar_passivo(fundo, pdf_path, corpo, texto_whats, data, email=True, whatsapp=True):
    """fundo: 'GARE' ou 'GAME'. Assunto: RELATÓRIO DE PASSIVO {fundo} DIA {data}."""
    if email:
        _enviar_email(pdf_path, corpo, f"RELATÓRIO DE PASSIVO {fundo} DIA {data}")
    if whatsapp:
        _enviar_whatsapp(texto_whats, pdf_path)

In [12]:
# ============================================================
# 4) 👀 PRÉ-VISUALIZAÇÃO — não envia nada. Rode antes das células 5 e 6.
# ============================================================
from IPython.display import HTML, display

for fundo, pdf_path, corpo, texto in [
    ("GARE", pdf_gare, CORPO_GARE, WHATS_GARE),
    ("GAME", pdf_game, CORPO_GAME, WHATS_GAME),
]:
    print("=" * 78)
    print(f"Assunto : RELATÓRIO DE PASSIVO {fundo} DIA {data}")
    print(f"Anexo   : {'✅' if os.path.exists(pdf_path) else '❌ NÃO ENCONTRADO'}  {pdf_path}")
    print(f"Grupo   : {GRUPO_PASSIVO_PHONE}")
    print("-" * 78)
    display(HTML(_html_email(corpo)))
    print("--- WhatsApp ".ljust(78, "-"))
    print(texto)
    print()

Assunto : RELATÓRIO DE PASSIVO GARE DIA 04.09.2026
Anexo   : ✅  g:\.shortcut-targets-by-id\1YZLLTFOuLniE6hO7WNHXl2c3MSa4TOv6\X. GAME11\GUARDIAN Institucional\5. Planejamento & Processos\2. 2023\4. Research\2025\5. Inteligência Passivos Guardian\2. RELATÓRIOS\1. DIARIOS\GARE\Passivos_GARE11_04.09.2026.pdf
Grupo   : 120363370142403605-group
------------------------------------------------------------------------------


--- WhatsApp -----------------------------------------------------------------
📅 GARE11 - 04/09/2026

⚠️ NECTON LIDERA SALDO NEGATIVO EM DIA DE QUEDA DA COTA CONTRA O IFIX COM TIVIO OPPORTUNITIES MANTENDO PRESSÃO VENDEDORA NA FAIXA 3M-15M

📈 Volume negociado foi de R$ 7,1 milhões. A cota fechou o dia em queda de -0,48%, fechando a R$ 8,34. O fundo registrou mais um dia de entrada líquida de cotistas, ultrapassando o patamar de 555 mil

📊 Desempenho do Dia:
* GARE11: -0,48%
* IFIX: 0,2%
* Daytrade: 42% do volume total

📈 Compras (TOP 10 – R$ 0,9 mi)
* 5k-20k: 30,79%
* 1k-5k: 18,96%
* 0-1k: 16,26%

🔹KINEA HEDGE FUND compra R$ 416k mantendo-se na faixa 500k-1M — maior compra do dia.
🔹5 novos cotistas (NC) ingressam no fundo totalizando R$ 255k, com destaque para ALAN NADER ACKEL GHANI com R$ 100k.
🔹Cotista da faixa 100k-500k compra R$ 97k mantendo-se na mesma faixa.

📉 Vendas (TOP 10 – R$ 2,6 mi)
* 3M-15M: 38,97%
* 5k-20k: 14,27%
* 50k-100k: 12,67%

🔹TIVIO OPPORTUNITIES MASTER FUNDO vende

--- WhatsApp -----------------------------------------------------------------
📅 GAME11 - 04/09/2026

⚠️ GUARDIAN STRATEGY FII COMPRA R$87K NO SÉTIMO PREGÃO CONSECUTIVO, COM MANAT RECEBÍVEIS E XTX MARKETS SOMANDO R$137K EM VENDAS INSTITUCIONAIS

📈 Volume negociado foi de R$ 0,26 milhões. A cota fechou o dia em alta de 0,9%, fechando a R$ 8,29. O fundo registrou saída líquida de cotistas (9 entradas x 16 saídas), mantendo-se no patamar de 20.244 cotistas.

📊 Desempenho do Dia:

* GAME11: 0,9%
* IFIX PAPEL: 0,1%
* Daytrade: 43% do volume total

📈 Compras (TOP 10 – R$ 129 mil)

* >15M: 59,27%
* 5k-20k: 17,51%
* 20k-50k: 11,00%

🔹GUARDIAN STRATEGY FII DE RL compra R$87k, mantendo-se na faixa >15M pelo sétimo pregão consecutivo; somado à RIGEL COVE LP, que adquiriu R$5k saindo da faixa 0-1k para 5k-20k, os institucionais respondem por R$92k das compras.
🔹Cotista da faixa 20k-50k compra R$11k e migra para a faixa 50k-100k, entrando em categoria de alta concentração financeira.
🔹Cotista da fa

In [13]:
# ============================================================
# 5) ENVIAR E-MAILS — confira a célula 4 antes de rodar
# ============================================================
enviar_passivo("GARE", pdf_gare, CORPO_GARE, WHATS_GARE, data, whatsapp=False)
enviar_passivo("GAME", pdf_game, CORPO_GAME, WHATS_GAME, data, whatsapp=False)

✅ E-mail enviado — RELATÓRIO DE PASSIVO GARE DIA 04.09.2026
✅ E-mail enviado — RELATÓRIO DE PASSIVO GAME DIA 04.09.2026


In [14]:
# ============================================================
# 6) ENVIAR AO WHATSAPP — grupo "Guardian - PASSIVO inteligência listados"
#    4 mensagens: texto GARE -> PDF GARE -> texto GAME -> PDF GAME
# ============================================================
enviar_passivo("GARE", pdf_gare, CORPO_GARE, WHATS_GARE, data, email=False)
enviar_passivo("GAME", pdf_game, CORPO_GAME, WHATS_GAME, data, email=False)

✅ Texto enviado ao WhatsApp (120363370142403605-group). messageId: 3EB02589FF09356EE167CB
✅ PDF enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0EA3CBF6BE32858ECB6
✅ Texto enviado ao WhatsApp (120363370142403605-group). messageId: 3EB005FBFE98F16AE0F4D5
✅ PDF enviado ao WhatsApp (120363370142403605-group). messageId: 3EB0938C0866D673B9F847
